# Exercise 40: Restaurant Growth

Table: `Customer`
```
+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| customer_id   | int     |
| name          | varchar |
| visited_on    | date    |
| amount        | int     |
+---------------+---------+
```
- `(customer_id, visited_on)` is the primary key.
- `visited_on` is the date the `customer_id` with name `name` visited a restaurant.
- `amount` is the total paid by the customer.

**Task.**
Compute the **average amount spent in the restaurant (by all customers) rounded to two decimals over seven days** (moving with time) from the given date together with the 6 preceding dates.
Return the result **ordered by** `visited_on` in ascending order.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_40 WITH (FORCE);
%sql CREATE DATABASE exercise_40;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_40 --alias exercise_40

Connecting and switching to connection 'exercise_40'

#### Fill the database with content

In [4]:
%%sql 

DROP TABLE IF EXISTS Customer;

CREATE TABLE Customer (
    customer_id INTEGER NOT NULL,
    name VARCHAR NOT NULL,
    visited_on DATE NOT NULL,
    amount INTEGER NOT NULL,
    PRIMARY KEY (customer_id, visited_on)
);

INSERT INTO Customer (customer_id, name, visited_on, amount)
VALUES
(1, 'Jhon', '2019-01-01', 100),
(2, 'Daniel', '2019-01-02', 110),
(3, 'Jade', '2019-01-03', 120),
(4, 'Khaled', '2019-01-04', 130),
(5, 'Winston', '2019-01-05', 110), 
(6, 'Elvis', '2019-01-06', 140), 
(7, 'Anna', '2019-01-07', 150),
(8, 'Maria', '2019-01-08', 80 ),
(9, 'Jaze', '2019-01-09', 110), 
(1, 'Jhon', '2019-01-10', 130), 
(3, 'Jade', '2019-01-10', 150);

Running query in 'exercise_40'

11 rows affected.

++
||
++
++

#### Solve the exercise

**Task.**
Compute the **average amount spent in the restaurant (by all customers) rounded to two decimals over seven days** (moving with time) from the given date together with the 6 preceding dates.
Return the result **ordered by** `visited_on` in ascending order.

In [5]:
%%sql
SELECT
    visited_on, amount, ROUND(amount::NUMERIC/7,2) AS average_amount
FROM (
    SELECT DISTINCT visited_on, SUM(amount) OVER (
        ORDER BY visited_on RANGE BETWEEN '6 days' PRECEDING AND CURRENT ROW
        ) AS amount
    FROM Customer
    )
WHERE visited_on >= (SELECT MIN(visited_on) FROM Customer) + INTERVAL '6 days'
ORDER BY visited_on

Running query in 'exercise_40'

4 rows affected.

visited_on,amount,average_amount
2019-01-07,860,122.86
2019-01-08,840,120.00
2019-01-09,840,120.00
2019-01-10,1000,142.86


#### Explanation

- There is a subquery summing the spent amount in a moving window of seven days. The sum over a moving window is done by `SUM(amount) OVER (ORDER BY ... BETWEEN '6 days' PRECEDING AND CURRENT ROW)`
- In the subquery one makes also sure that for every date there is at most one entry using `DISTINCT visited_on`.
- Then take a `ROUND(amount::NUMERIC/7,2)` to find the average and round it to two decimals.
- Moreover, the `WHERE` clause makes sure that one start from the seventh entry using a subquery which adds `6 days` to `MIN(visited_on)`.
- Finally, order by `visited_on`.

#### Close connection

In [6]:
%sql --close exercise_40
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_40;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)